# **Feature Engineering**

## Objectives

- Prepare the cleaned Premier League player dataset for machine learning.
- Select appropriate predictor features based on findings from exploratory data analysis.
- Remove features that would cause target leakage.
- Investigate and handle missing values required for modelling.
- Transform categorical variables into a suitable numerical representation.
- Prepare and save the engineered dataset for model development.

## Inputs

- `data/processed/all_players_cleaned.csv` – Cleaned player-season dataset produced by the Data Cleaning notebook.

## Outputs

- `data/processed/engineered_players.csv` – Feature-engineered dataset containing the selected predictors and `HighScorer` target for model development.

## Additional Comments

- The `HighScorer` target represents players who scored 10 or more Premier League goals in a season.
- EDA identified substantial class imbalance, correlated attacking statistics and systematic missing values in several shooting features. These findings will be considered when preparing the features for modelling.

### Imports

In [1]:
import os
import pandas as pd
import numpy as np

### Change Working directory

In [2]:
current_dir = os.getcwd()
current_dir

'c:\\code\\premier-league-predictor\\premier-league-predictor\\jupyter_notebooks'

In [3]:
os.chdir(r"C:\code\premier-league-predictor\premier-league-predictor")
print("You set a new current directory")

You set a new current directory


In [4]:
current_dir = os.getcwd()
current_dir

'C:\\code\\premier-league-predictor\\premier-league-predictor'

In [5]:
df = pd.read_csv("data/processed/all_players_cleaned.csv")

In [6]:
df.shape

(8196, 54)

In [7]:
df.head()

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77.0,NaN,0.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78.0,NaN,10.0,32.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83.0,0.0,1.0,23.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16


## Define Target Variable

The machine learning model will predict whether a player is classified as a `HighScorer`, defined as scoring 10 or more Premier League goals in a season.

The target variable is recreated from `Goals` before goal-derived features are removed from the predictor dataset.

In [8]:
df.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Goals',
       'Headed goals', 'Goals with right foot', 'Goals with left foot',
       'Hit woodwork', 'Goals per match', 'Penalties scored',
       'Freekicks scored', 'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed', 'Saves', 'Penalties saved', 'Punches',
       'High Claims', 'Catches', 'Sweeper clearances', 'Throw outs',
       'Goal Kicks', 'Season'],
     

In [9]:
df["HighScorer"] = df["Goals"] >= 10

In [10]:
df["HighScorer"].value_counts()

HighScorer
False    7984
True      212
Name: count, dtype: int64

### Remove Target Leakage Features

The `HighScorer` target is derived from the number of goals scored by each player. Therefore, `Goals` cannot be used as a predictor because it would directly reveal information used to determine the target.

Other statistics that directly describe how those goals were scored are also excluded to reduce target leakage. The model should instead learn from player characteristics and performance statistics that do not directly provide the outcome being predicted.

In [11]:
goal_columns = [
    "Goals",
    "Headed goals",
    "Goals with right foot",
    "Goals with left foot",
    "Goals per match",
    "Penalties scored",
    "Freekicks scored"
]

In [12]:
goal_columns

['Goals',
 'Headed goals',
 'Goals with right foot',
 'Goals with left foot',
 'Goals per match',
 'Penalties scored',
 'Freekicks scored']

In [13]:
x = df.drop(columns=goal_columns + ["HighScorer"])
y = df["HighScorer"]

In [14]:
x.shape

(8196, 47)

In [15]:
y.shape

(8196,)

In [16]:
x.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Hit woodwork',
       'Shots', 'Shots on target', 'Shooting accuracy %', 'Big chances missed',
       'Saves', 'Penalties saved', 'Punches', 'High Claims', 'Catches',
       'Sweeper clearances', 'Throw outs', 'Goal Kicks', 'Season'],
      dtype='object')

### Target and Leakage Observations

The `HighScorer` target contains **212 high-scorer records** and **7,984 non-high-scorer records** across 8,196 player-season observations.

Seven goal-derived features were excluded from the predictor dataset to prevent target leakage. The target variable was also separated from the predictors, resulting in **47 candidate predictor features**.

Further feature selection is required before modelling, as not all remaining features are expected to provide useful information for predicting high scorers.

***

## Feature Selection

The predictor dataset currently contains 47 candidate features. Exploratory data analysis showed that not all available statistics are equally relevant to identifying high scorers.

A smaller set of candidate features will therefore be selected based on their relationship with the target, footballing relevance and findings from the exploratory analysis.

In [17]:
x = x.drop(columns=["Name"])

In [18]:
x.shape

(8196, 46)

`Name` is excluded because it is an identifier rather than a generalisable player performance feature.

`Season` is also excluded. Exploratory analysis showed that the proportion of high scorers remained relatively consistent across seasons, and the season itself is not considered necessary for predicting whether a player's performance statistics indicate a high scorer.

In [19]:
x = x.drop(columns=["Season"])

In [20]:
x.shape

(8196, 45)

***

### Remove Goalkeeper-Specific Features

The dataset contains several statistics that are specific to goalkeepers, including saves, penalties saved, punches, claims, catches and distribution statistics.

These features are removed because they describe goalkeeper-specific actions rather than attacking performance. Including them would primarily help the model identify goalkeepers rather than provide meaningful information for predicting whether a player will be classified as a high scorer.

The `Position` feature is retained separately because EDA showed a clear relationship between playing position and the `HighScorer` target.

In [21]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks"
]

In [22]:
x = x.drop(columns=goalkeeper_columns)

In [23]:
x.shape

(8196, 37)

In [24]:
x.columns

Index(['Position', 'Appearances', 'Clean sheets', 'Goals conceded', 'Tackles',
       'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Hit woodwork',
       'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed'],
      dtype='object')

***

### Select Candidate Features

Based on the findings from exploratory data analysis, a smaller set of candidate features is selected for further preparation and modelling.

The selected features focus primarily on playing position, appearances, attacking performance, creativity and player involvement. Defensive and lower-relevance statistics are excluded to reduce unnecessary complexity.

`Passes per match` is retained instead of total `Passes`. Total passes are influenced by the number of appearances a player makes, while passes per match provides a better indication of typical passing involvement. `Appearances` is retained separately to represent playing time.

These features represent an initial EDA-informed selection and may be refined further when considering missing values, correlation and model performance.

In [25]:
selected_features = [
    "Position",
    "Appearances",
    "Blocked shots",
    "Assists",
    "Passes per match",
    "Big chances created",
    "Crosses",
    "Cross accuracy %",
    "Through balls",
    "Offsides",
    "Hit woodwork",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
    "Recoveries",
    "Duels won",
    "Duels lost",
    "Successful 50/50s",
    "Aerial battles won",
    "Aerial battles lost"
]

In [26]:
x = x[selected_features]

In [27]:
x.shape

(8196, 21)

In [28]:
x.head()

,Position,Appearances,Blocked shots,Assists,Passes per match,Big chances created,Crosses,Cross accuracy %,Through balls,Offsides,...,Shots,Shots on target,Shooting accuracy %,Big chances missed,Recoveries,Duels won,Duels lost,Successful 50/50s,Aerial battles won,Aerial battles lost
0,Midfielder,10,0.0,1,11.90,1.0,12.0,25.0,0.0,0.0,...,2.0,1.0,50.0,0.0,23.0,29.0,34.0,5.0,4.0,6.0
1,Midfielder,32,10.0,0,29.31,4.0,55.0,31.0,2.0,1.0,...,39.0,10.0,26.0,1.0,137.0,140.0,153.0,12.0,22.0,31.0
2,Defender,15,1.0,1,35.07,0.0,31.0,16.0,0.0,2.0,...,NaN,NaN,NaN,NaN,76.0,67.0,65.0,8.0,8.0,13.0
3,Midfielder,0,0.0,0,0.00,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Forward,2,1.0,0,5.00,0.0,2.0,NaN,NaN,3.0,...,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Value Assessment

The selected feature set is assessed for missing values before any transformations are applied. Missing values were identified during EDA, but they are reassessed here because only the features selected for modelling now need to be considered.

The percentage of missing values in each selected feature is calculated to help determine an appropriate treatment strategy.

In [29]:
x.isna().mean() * 100

Position                0.000000
Appearances             0.000000
Blocked shots          11.725232
Assists                 0.000000
Passes per match        0.000000
Big chances created     0.000000
Crosses                11.725232
Cross accuracy %       32.613470
Through balls          32.613470
Offsides               11.725232
Hit woodwork            0.000000
Shots                  43.643241
Shots on target        43.643241
Shooting accuracy %    43.643241
Big chances missed     43.643241
Recoveries             32.613470
Duels won              32.613470
Duels lost             32.613470
Successful 50/50s      32.613470
Aerial battles won     32.613470
Aerial battles lost    32.613470
dtype: float64

In [30]:
shooting_columns = [
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed"
]

In [31]:
for column in shooting_columns:
    print(column, ((df["HighScorer"]) & (df[column].isna())).sum())


Shots 0
Shots on target 0
Shooting accuracy % 0
Big chances missed 0


### Missing Value Considerations

EDA identified systematic missingness in several selected features, particularly the shooting statistics. These features are retained because they showed strong relationships with the `HighScorer` target despite their missing values.

A further check confirmed that none of the 212 high scorers have missing values for `Shots`, `Shots on target`, `Shooting accuracy %` or `Big chances missed`.

The remaining missing values will therefore be handled during feature preparation rather than removing these potentially informative features.

***

### Refine Selected Features

Several player involvement features contain approximately 32.6% missing values. Although EDA showed higher values among high scorers, these statistics are cumulative season totals and were also associated with greater playing time.

To reduce unnecessary complexity and avoid imputing a large amount of missing data for lower-priority predictors, these involvement features are excluded from the candidate feature set.

In [32]:
involvement_columns = [
    "Recoveries",
    "Duels won",
    "Duels lost",
    "Successful 50/50s",
    "Aerial battles won",
    "Aerial battles lost"
]

In [33]:
x = x.drop(columns=involvement_columns)

In [34]:
x.shape

(8196, 15)

The removal of the six lower-priority involvement features reduced the candidate feature set from 21 to 15 features. The remaining features focus more directly on playing position, appearances, attacking activity and creative involvement.

In [35]:
creative_columns = [
    "Cross accuracy %",
    "Through balls"
]

In [36]:
for column in creative_columns:
    print(column, ((df["HighScorer"]) & (df[column].isna())).sum())

Cross accuracy % 171
Through balls 171


### Creative Feature Missingness

`Cross accuracy %` and `Through balls` were investigated separately because both contain approximately 32.6% missing values.

Of the 212 high scorers, **171 have missing values for each of these features**. As these features are unavailable for the majority of the positive target class, retaining them would require substantial imputation and may provide limited reliable information for identifying high scorers.

Both features are therefore excluded from the candidate feature set.

In [37]:
x = x.drop(columns=creative_columns)


In [38]:
x.shape

(8196, 13)

In [39]:
x.columns

Index(['Position', 'Appearances', 'Blocked shots', 'Assists',
       'Passes per match', 'Big chances created', 'Crosses', 'Offsides',
       'Hit woodwork', 'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed'],
      dtype='object')

In [40]:
x.isna().mean() * 100

Position                0.000000
Appearances             0.000000
Blocked shots          11.725232
Assists                 0.000000
Passes per match        0.000000
Big chances created     0.000000
Crosses                11.725232
Offsides               11.725232
Hit woodwork            0.000000
Shots                  43.643241
Shots on target        43.643241
Shooting accuracy %    43.643241
Big chances missed     43.643241
dtype: float64

In [41]:
missing_columns = [
    "Blocked shots",
    "Crosses",
    "Offsides"
]

In [42]:
for column in missing_columns:
    print(column, ((df["HighScorer"]) & (df[column].isna())).sum())

Blocked shots 0
Crosses 0
Offsides 0


### Missing Value Treatment

The remaining features with missing values were investigated against the target variable. None of the 212 high scorers have missing values in the retained shooting, crossing, offside or blocked-shot features.

Rows containing missing values will not be removed, as this would result in the loss of a substantial proportion of the dataset. Missing numerical values will instead be imputed during model preparation.

Imputation will be fitted using the training data only to prevent information from the test set influencing the preprocessing process.

***

## Encode Categorical Features

`Position` is the remaining categorical feature in the selected predictor dataset.

Playing position is retained because EDA identified a clear relationship between position and the `HighScorer` target. However, the categorical position labels cannot be used directly by most machine learning algorithms.

One-hot encoding will therefore be used to represent each playing position without introducing an artificial numerical order between the categories.

In [43]:
x["Position"].value_counts()

Position
Midfielder    2877
Defender      2634
Forward       1724
Goalkeeper     961
Name: count, dtype: int64

In [44]:
x = pd.get_dummies(x, columns=["Position"], dtype=int)

In [45]:
x.shape

(8196, 16)

In [46]:
x.head()

,Appearances,Blocked shots,Assists,Passes per match,Big chances created,Crosses,Offsides,Hit woodwork,Shots,Shots on target,Shooting accuracy %,Big chances missed,Position_Defender,Position_Forward,Position_Goalkeeper,Position_Midfielder
0,10,0.0,1,11.90,1.0,12.0,0.0,0.0,2.0,1.0,50.0,0.0,0,0,0,1
1,32,10.0,0,29.31,4.0,55.0,1.0,1.0,39.0,10.0,26.0,1.0,0,0,0,1
2,15,1.0,1,35.07,0.0,31.0,2.0,0.0,NaN,NaN,NaN,NaN,1,0,0,0
3,0,0.0,0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,1
4,2,1.0,0,5.00,0.0,2.0,3.0,0.0,2.0,0.0,0.0,0.0,0,1,0,0


### Encoding Observation

One-hot encoding replaced the original `Position` feature with four numerical position features. This increased the predictor dataset from 13 to 16 features while preserving the playing-position information without introducing an artificial numerical order.

In [47]:
x.isna().sum()

Appearances               0
Blocked shots           961
Assists                   0
Passes per match          0
Big chances created       0
Crosses                 961
Offsides                961
Hit woodwork              0
Shots                  3577
Shots on target        3577
Shooting accuracy %    3577
Big chances missed     3577
Position_Defender         0
Position_Forward          0
Position_Goalkeeper       0
Position_Midfielder       0
dtype: int64

In [48]:
x.dtypes

Appearances              int64
Blocked shots          float64
Assists                  int64
Passes per match       float64
Big chances created    float64
Crosses                float64
Offsides               float64
Hit woodwork           float64
Shots                  float64
Shots on target        float64
Shooting accuracy %    float64
Big chances missed     float64
Position_Defender        int64
Position_Forward         int64
Position_Goalkeeper      int64
Position_Midfielder      int64
dtype: object

In [49]:
y.isna().sum()

np.int64(0)

In [50]:
y.value_counts()

HighScorer
False    7984
True      212
Name: count, dtype: int64

***

## Feature Transformation Investigation

A rate-based feature is investigated to determine whether adjusting a cumulative statistic for playing time could provide useful additional information.

`Assists` is a cumulative season statistic and is influenced by the number of appearances a player makes. An `Assists per Appearance` feature will therefore be investigated to represent a player's assisting contribution relative to their playing time.

The transformed feature will only be retained if it provides useful information beyond the existing features.

In [51]:
x.shape

(8196, 16)

In [52]:
((x["Appearances"] == 0) & (x["Assists"] > 0)).sum()

np.int64(0)

In [53]:
x["Assists per Appearance"] = np.where(
    x["Appearances"] == 0,
    0,
    x["Assists"] / x["Appearances"]
)

In [54]:
x[["Appearances", "Assists", "Assists per Appearance"]].head(10)

,Appearances,Assists,Assists per Appearance
0,10,1,0.100000
1,32,0,0.000000
2,15,1,0.066667
3,0,0,0.000000
4,2,0,0.000000
5,22,1,0.045455
6,12,1,0.083333
7,32,0,0.000000
8,31,2,0.064516
9,15,0,0.000000


In [55]:
x.groupby(y)["Assists per Appearance"].agg(["mean", "median"])

,mean,median
HighScorer,,
False,0.026757,0.000000
True,0.154412,0.142857


### Transformation Observation

The engineered `Assists per Appearance` feature shows a clear difference between the target classes. High scorers have a mean of approximately **0.154 assists per appearance**, compared with approximately **0.027** for non-high-scorers.

The median is also higher for high scorers (**0.143**) than for non-high-scorers (**0.000**).

This suggests that assists relative to appearances may provide useful information for identifying high scorers. The engineered feature is therefore retained for modelling.

***

In [56]:
engineered_players = x.copy()
engineered_players["HighScorer"] = y

In [57]:
engineered_players.shape

(8196, 18)

## Save Feature-Engineered Dataset

The selected and encoded predictor features are recombined with the `HighScorer` target and saved for use in the machine learning modelling stage.

Missing numerical values are intentionally retained at this stage. Imputation will be fitted using the training data after the train/test split to prevent information from the test set influencing preprocessing.

In [58]:
output_path = "data/processed/engineered_players.csv"
engineered_players.to_csv(output_path, index=False)
print(f"Dataset saved to {output_path}")

Dataset saved to data/processed/engineered_players.csv


## Conclusions

Feature engineering reduced the original dataset to a focused set of predictors for the `HighScorer` classification problem.

Target leakage and lower-relevance features were removed, playing position was one-hot encoded, and an `Assists per Appearance` feature was engineered and retained after showing a clear difference between the target classes.

The final dataset contains **8,196 player-season records, 17 predictor features and the `HighScorer` target**. Missing numerical values have been intentionally retained for imputation after the train/test split during model development.